# Safety computPtions

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np

from exp_mpc.stewart_min import mpc_spec

jax.config.update("jax_enable_x64", True)

## Fast 6x6 matrix inversion

JAX's implementation of a matrix inverse requires LAPACK by default on CPU (it uses LAPACK to solve several linear systems).
This is reasonable, but my C++ exporting software doesn't link to this by default.
Also, the default matrix inversion code is apparently slow: https://github.com/jax-ml/jax/issues/11321.
We implement our own custom matrix inversion code (using the Schur complement) for $6 \times 6$ matrices.
It isn't obvious that inverting the matrix is worse than solving the one linear system that we are interested in, in practice.
I should test this against LAPACK sometime, out of curiosity.

In [ ]:
def det3(P):
    t0 = P[0, 0] * (P[1, 1] * P[2, 2] - P[1, 2] * P[2, 1])
    t1 = P[1, 0] * (P[2, 1] * P[0, 2] - P[0, 1] * P[2, 2])
    t2 = P[2, 0] * (P[0, 1] * P[1, 2] - P[0, 2] * P[1, 1])
    return t0 + t1 + t2

P = np.random.uniform(-1, 1, size=9).reshape(3, 3)
assert np.allclose(det3(P), np.linalg.det(P))

In [ ]:
def inv3(P):
    cross = jnp.vstack([
        jnp.linalg.cross(P[:, 1], P[:, 2]).reshape(1, -1),
        jnp.linalg.cross(P[:, 2], P[:, 0]).reshape(1, -1),
        jnp.linalg.cross(P[:, 0], P[:, 1]).reshape(1, -1),
    ])
    return cross / det3(P)

P = np.random.uniform(-1, 1, size=9).reshape(3, 3)
assert np.allclose(det3(P), np.linalg.det(P))

In [ ]:
@jax.custom_jvp
@jax.jit
def inv6(P):
    assert P.shape == (6, 6)
    A = P[:3, :3]
    B = P[:3, 3:]
    C = P[3:, :3]
    D = P[3:, 3:]

    A3 = inv3(A)
    CA3 = C @ A3
    A3B = A3 @ B
    schur = inv3(D - CA3 @ B)
    schur_CA3 = schur @ CA3
    return jnp.vstack([
        jnp.hstack([A3 + A3B @ schur_CA3, -A3B @ schur]),
        jnp.hstack([-schur_CA3, schur]),
    ])

@inv6.defjvp
def _inv6_jvp(primals, tangents):
    P, = primals
    P_dot, = tangents
    P_inv = inv6(P)
    return P_inv, -P_inv @ P_dot @ P_inv

P = np.random.uniform(-1, 1, size=36).reshape(6, 6)
assert jnp.allclose(inv6(P), np.linalg.inv(P))
assert jnp.allclose(jax.jacrev(inv6)(P), jax.jacrev(jnp.linalg.inv)(P))

## leg safety

Only compute the final change in leg length.
We assume that this is the only important factor in our MPC control law.
See SMS's notes.

In [ ]:
def velocity_jacobian(t: jax.Array, u: jax.Array) -> jax.Array:
    assert t.shape == (6, 3) and u.shape == (6, 3)
    t_cross_u = jnp.linalg.cross(t, u, axis=1)
    return jnp.hstack([u, t_cross_u])

def comp_a_f(
    gravity: jax.Array,
    tops: jax.Array,
    m_t: jax.Array,
    m_r: jax.Array,
    r_0: jax.Array,
    u: jax.Array,
) -> jax.Array:
    lam_inv = inv6(velocity_jacobian(tops, u))
    twist = jnp.concatenate([gravity, jnp.cross(r_0, gravity)])
    m = m_t / m_r
    a_f = m * lam_inv.T @ twist
    return a_f


def comp_ell_M(
    gravity: jax.Array,
    m_t: jax.Array,
    m_r: jax.Array,
    t_e: jax.Array,
    a_b: jax.Array,
    tops: jax.Array,
    r_0: jax.Array,
    u: jax.Array,
    ell_0: jax.Array,
    v_0: jax.Array,
) -> jax.Array:
    assert r_0.shape == (3,)
    assert u.shape == (6, 3)
    assert ell_0.shape == (6,)
    assert v_0.shape == (6,)

    # note that jnp.sign(v_e) is defined to have zero gradient everywhere
    # this is the desired behavior for this function

    a_f = comp_a_f(gravity, tops, m_t, m_r, r_0, u)
    v_e = v_0 + a_f * t_e
    a_n = a_f - jnp.sign(v_e) * a_b
    delta_ell = -(v_e**2) / (2 * a_n)
    ell_M = ell_0 + v_0 * t_e + 0.5 * a_f * t_e**2
    return ell_M + delta_ell


def comp_ell_M_utils(
    spec: mpc_spec.MPCSpec,
    r_0: jax.Array,
    u: jax.Array,
    ell_0: jax.Array,
    v_0: jax.Array,
) -> jax.Array:
    return comp_ell_M(-mpc_spec.gravity, spec.m_t, spec.m_r, spec.t_e, spec.a_b, spec.tops, r_0, u, ell_0, v_0)

In [ ]:
spec = mpc_spec.MPCSpec()
r_0 = spec.r_0
u = (spec.tops - spec.bots)
norms = np.linalg.norm(spec.tops - spec.bots, axis=1).reshape(-1, 1)
u /= np.tile(norms, reps=(1, 3))
ell_0 = np.ones(6) * spec.lengths_home
v_0 = np.zeros(6)
comp_ell_M_utils(spec, r_0, u, ell_0, v_0) - ell_0